# V-Max BC training on Colab

Runs `algorithm=bc` training from `as-fast-as-anyone` (V-Max fork) on a Colab GPU instead of the local GTX 1660 Super (6GB VRAM / 15GB RAM).

**Before running this notebook**, upload to your Google Drive:
```
MyDrive/vmax_workdir/data/train_91f.tar                                (local: tar of data/train_91f/, 178GB)
MyDrive/vmax_workdir/data/scores/combined_scores.csv                   (local: data/scores/combined_scores.csv, 6.3MB)
MyDrive/vmax_workdir/data/eval/val_sample_shards_hanam/                (local: data/eval/val_sample_shards_hanam/, 1.3MB)
```
`data/shards/*` (the hard/easy pools etc.) are **not** uploaded - they're just symlinks into `train_91f`, so re-running `split_hard_easy_pools.py` + `merge_pools.py` on Colab regenerates them in seconds once `train_91f` + the scores CSV are there.

**Upload as a single tar, not the raw folder**: `train_91f` is 58,925 individual files - uploading that as-is (browser drag/drop, or a naive Drive sync) is extremely slow and prone to failing partway through. Tar it into ONE file first. Since the local disk here doesn't have 178GB spare to hold a second copy of the tar, don't build it locally then upload - stream it straight into wherever it's headed, e.g. with `rclone` configured for your Drive:
```bash
tar cf - -C /home/ehdtod001009/Downloads/dxchallenge_motion_planning data/train_91f | \
  rclone rcat remotename:vmax_workdir/data/train_91f.tar
```
(or, if you use the Google Drive desktop app, `tar cf <mounted-drive-path>/vmax_workdir/data/train_91f.tar -C .../dxchallenge_motion_planning data/train_91f` writes straight into the synced folder - ask if you want help setting either of these up.)

Runtime > Change runtime type > select a GPU (T4 is free-tier; Colab Pro gives A100/L4).

**Why Drive at all**: Colab sessions disconnect (idle timeout / max runtime). Checkpoints are written straight to Drive (via a symlink), and BC training now supports full resume - if the session dies, just re-run the training cell with the same `name_run` and it picks up from the last checkpoint instead of restarting.

In [ ]:
!nvidia-smi

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_WORKDIR = "/content/drive/MyDrive/vmax_workdir"
import os
for required in [
    f"{DRIVE_WORKDIR}/data/train_91f.tar",
    f"{DRIVE_WORKDIR}/data/scores/combined_scores.csv",
    f"{DRIVE_WORKDIR}/data/eval/val_sample_shards_hanam",
]:
    assert os.path.exists(required), f"Missing {required} - upload it first (see markdown above)."
print("Drive OK, everything needed is there.")

## 2. Clone the repo and set up the environment (uv, pinned by uv.lock)

In [ ]:
%cd /content
!rm -rf as-fast-as-anyone
!git clone https://github.com/gm2256/as-fast-as-anyone.git
%cd /content/as-fast-as-anyone/V-Max

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Installs its own Python 3.12 (per .python-version) regardless of Colab's system Python,
# and resolves the exact versions pinned in uv.lock (same env as the local machine).
!uv sync

## 3. Extract train_91f and rebuild the hard/easy pools

Extracts straight from the Drive-mounted tar (one large sequential read over the FUSE mount - much faster than copying 58,925 loose files off Drive first) so only one 178GB copy ever sits on Colab's local disk. Checks free space first since a fresh T4 runtime's disk isn't always $>$178GB.

In [ ]:
!df -h /content
!mkdir -p /content/data
!tar -xf "$DRIVE_WORKDIR/data/train_91f.tar" -C /content/data
!du -sh /content/data/train_91f

In [ ]:
!mkdir -p /content/data/scores
!cp "$DRIVE_WORKDIR/data/scores/combined_scores.csv" /content/data/scores/combined_scores.csv

%cd /content/as-fast-as-anyone/V-Max
# Regenerates the exact same per-site hard/easy split as the local machine
# (--hard-frac 0.4 is what produced the currently-committed local pools).
!uv run python scripts/split_hard_easy_pools.py \
    /content/data/scores/combined_scores.csv /content/data/train_91f /content/data/shards/mixture_pools --hard-frac 0.4
!uv run python scripts/merge_pools.py /content/data/shards/bc_pools \
    hard=/content/data/shards/mixture_pools/hanam_hard,/content/data/shards/mixture_pools/jeju_hard \
    easy=/content/data/shards/mixture_pools/hanam_easy,/content/data/shards/mixture_pools/jeju_easy

## 4. Wire up checkpoints (Drive, persistent)

`runs/` is symlinked into Drive so checkpoints/logs survive a disconnect.

In [ ]:
import os
os.makedirs(f"{DRIVE_WORKDIR}/runs", exist_ok=True)
!rm -rf /content/as-fast-as-anyone/V-Max/runs
!ln -s "$DRIVE_WORKDIR/runs" /content/as-fast-as-anyone/V-Max/runs
!ls -la /content/as-fast-as-anyone/V-Max/runs

## 5. Train

`@23570` / `@35355` should match the counts `merge_pools.py` printed above (same as the local pools, since it's the same 178GB dataset and the same `--hard-frac`) - edit if they differ.

`total_timesteps=5_000_000` is roughly one pass over the combined hard+easy pool (~59k scenarios x 80 steps). Bump it up (e.g. `20_000_000`, the framework's own default scale) once you've confirmed `train/imitation_loss` in TensorBoard is still trending down at 5M and want to keep going.

**If the session disconnects mid-run**: just re-run this cell unchanged (steps 3-4 too, to rebuild local data/pools, then this). `algorithm.resume=true` (default) picks up from `runs/<name_run>/model/train_state_latest.pkl` on Drive.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python vmax/scripts/training/train.py \
  algorithm=bc network/encoder=lq \
  total_timesteps=5_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=20000 \
  waymo_dataset=true \
  'mixture_datasets=[{path: /content/data/shards/bc_pools/hard/hard.tfrecord@23570, weight: 0.3}, {path: /content/data/shards/bc_pools/easy/easy.tfrecord@35355, weight: 0.7}]' \
  name_run=colab_bc_run1 log_freq=50 save_freq=1500

## 6. Watch training in TensorBoard (optional, run in a separate cell while training runs)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs

## 7. After training: sweep checkpoints on the fixed held-out set

In [ ]:
!mkdir -p /content/data/eval
!cp -r "$DRIVE_WORKDIR/data/eval/val_sample_shards_hanam" /content/data/eval/val_sample_shards_hanam

%cd /content/as-fast-as-anyone/V-Max
!uv run python scripts/evaluate_checkpoints.py \
  --name_run colab_bc_run1 \
  --path_dataset /content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300 \
  --waymo_dataset true --batch_size 4